In [1]:
# imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [2]:
# load data
DATA_PATH = "/mnt/DataVol/Beratungen/Yurttas/survival/data"

data_df = pd.read_csv(f"{DATA_PATH}/Excel table survival analysis.csv", sep="\t", encoding="utf-8")
data_df.head()

,Bday,OPDate,Sex,Zn HIPEC,Age,Tumor,Histo,T,N,M,...,CC,Tod Datum,Overall survival,Overall survival (months),Rezidiv,Lok Rezidiv,Datum Rezidiv,Recurrence-free survival,Recurrence-free survival (months),SF Grund
0,2/25/1991,3/10/2008,2,1,17,1,3,4,2,1,...,1,8/16/2008,"0 years, 5 months, 6 days",5.20,3,1,8/12/2008,"0 years, 5 months, 2 days",5.07,0
1,7/17/1993,7/14/2015,1,1,21,1,3,4,1,1,...,0,5/6/2018,"2 years, 9 months, 22 days",33.72,0,1,12/22/2016,"1 years, 5 months, 8 days",17.26,0
2,6/8/1995,11/23/2018,2,1,23,4,2,3,1,1,...,1,99,99,NaN,3,1,6/22/2022,"3 years, 6 months, 30 days",42.99,0
3,9/2/1979,11/15/2005,1,1,26,10,1,99,99,1,...,0,5/11/2009,"3 years, 5 months, 26 days",41.85,0,1.8,4/12/2006,"0 years, 4 months, 28 days",4.92,0
4,11/18/1990,12/29/2017,1,1,27,3,3,4,2,1,...,0,11/3/2019,"1 years, 10 months, 5 days",22.16,0,1,7/19/2018,"0 years, 6 months, 20 days",6.66,0


In [3]:
# selecting columns to use for analysis
cols_to_use = [
    "Bday", "OPDate", "Sex", "Age", "Tumor","sPCI", "pPCI", "Tod Datum", "Datum Rezidiv"
]

reduc_df = data_df[cols_to_use].copy()
reduc_df.head()

,Bday,OPDate,Sex,Age,Tumor,sPCI,pPCI,Tod Datum,Datum Rezidiv
0,2/25/1991,3/10/2008,2,17,1,26,22,8/16/2008,8/12/2008
1,7/17/1993,7/14/2015,1,21,1,5,3,5/6/2018,12/22/2016
2,6/8/1995,11/23/2018,2,23,4,15,6,99,6/22/2022
3,9/2/1979,11/15/2005,1,26,10,15,6,5/11/2009,4/12/2006
4,11/18/1990,12/29/2017,1,27,3,18,4,11/3/2019,7/19/2018


In [4]:
# converting date columns to datetime format
reduc_df["Bday"] = pd.to_datetime(reduc_df["Bday"], errors="coerce")
reduc_df["OPDate"] = pd.to_datetime(reduc_df["OPDate"], errors="coerce")
reduc_df["Tod Datum"] = pd.to_datetime(reduc_df["Tod Datum"], errors="coerce")
reduc_df["Datum Rezidiv"] = pd.to_datetime(reduc_df["Datum Rezidiv"], errors="coerce")
reduc_df.head()

,Bday,OPDate,Sex,Age,Tumor,sPCI,pPCI,Tod Datum,Datum Rezidiv
0,1991-02-25,2008-03-10,2,17,1,26,22,2008-08-16,2008-08-12
1,1993-07-17,2015-07-14,1,21,1,5,3,2018-05-06,2016-12-22
2,1995-06-08,2018-11-23,2,23,4,15,6,NaT,2022-06-22
3,1979-09-02,2005-11-15,1,26,10,15,6,2009-05-11,2006-04-12
4,1990-11-18,2017-12-29,1,27,3,18,4,2019-11-03,2018-07-19


In [5]:
# Time variable
reduc_df["time"] = np.where(
    reduc_df["Tod Datum"].notna(),
    (reduc_df["Tod Datum"] - reduc_df["OPDate"]).dt.days,
    np.where(
        reduc_df["Datum Rezidiv"].notna(),
        (reduc_df["Datum Rezidiv"] - reduc_df["OPDate"]).dt.days,
        np.nan
    )
)
reduc_df['months'] = reduc_df['time'] / 30.44  # Convert days to months

# Event indicator
reduc_df["event"] = np.where(reduc_df["Tod Datum"].notna(), 1, 0)

In [6]:
reduc_df["Age_calc"] = (reduc_df["OPDate"] - reduc_df["Bday"]).dt.days / 365.25

reduc_df["Age_final"] = reduc_df["Age"]
reduc_df.loc[reduc_df["Age_final"].isna(), "Age_final"] = reduc_df["Age_calc"]

reduc_df["Sex"] = reduc_df["Sex"]-1

In [7]:
reduc_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 285 entries, 0 to 284
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Bday           285 non-null    datetime64[ns]
 1   OPDate         285 non-null    datetime64[ns]
 2   Sex            285 non-null    int64         
 3   Age            285 non-null    int64         
 4   Tumor          285 non-null    int64         
 5   sPCI           285 non-null    int64         
 6   pPCI           285 non-null    int64         
 7   Tod Datum      131 non-null    datetime64[ns]
 8   Datum Rezidiv  263 non-null    datetime64[ns]
 9   time           285 non-null    float64       
 10  months         285 non-null    float64       
 11  event          285 non-null    int64         
 12  Age_calc       285 non-null    float64       
 13  Age_final      285 non-null    int64         
dtypes: datetime64[ns](4), float64(3), int64(7)
memory usage: 31.3 KB


In [8]:
# there are very rare tumor types, we will group them into "Other" category
reduc_df["Tumor"].value_counts()

Tumor
1     99
4     48
6     31
5     23
3     22
2     18
7     13
8     10
9      5
16     4
11     3
17     2
18     2
13     2
10     1
19     1
15     1
Name: count, dtype: int64

In [9]:
# Define frequent tumors
frequent = [1, 2, 3, 4, 5, 6, 7, 8]

reduc_df["Tumor_grouped"] = reduc_df["Tumor"].apply(
    lambda x: str(x) if x in frequent else "Other"
)

In [10]:
reduc_df = pd.get_dummies(reduc_df, columns=["Tumor_grouped"], drop_first=True)
reduc_df = pd.get_dummies(reduc_df, columns=["Sex"], drop_first=True)
cancer_dummies = [c for c in reduc_df.columns if c.startswith("Tumor_grouped_")]

In [11]:
reduc_df.to_csv(f"{DATA_PATH}/GPT_processed_survival_data.csv", index=False)